<table>
  <tr>
    <td><div align="left"><font size="30">Two-tank system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic coupled two-tank system using *bdsim***

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

The dynamics of this system are covered in many sources including: *Continuous System Simulation, D. J. Murray-Smith, Springer 1994,  ISBN 978-0-412-45150-8.  Chapter 10: Case study I - a two-tank liquid level control system.*

The system comprises two cylindrical tanks of water. A pump with a controllable flow rate (system input) fills the first tank.
A manually adjustable valve connects the two tanks,
and another manually adjustable valve drains water from the second tank.  The aim is to control the height of 
water in the second tank.

The flow rate through a valve is given by
$$
Q = \gamma C_{d} A \sqrt{2 g \Delta h}
$$
where $\gamma$ is the valve opening [0,1],  $C_{d}$ is the discharge coefficient of the valve (typically between 0.6 and 0.8, accounting for turbulence and friction), $A$ is the maximum cross-sectional area of the valve opening, $g$ is gravitational acceleration, and $\Delta h$ is the difference in *head* across the valve.

The water height in the two tanks can be written in terms of coupled first-order ODEs
$$
\begin{aligned}
\frac{d h_1}{dt} &= \frac{Q_i - Q_{12}}{A_1}\\ 
\frac{d h_2}{dt} &= \frac{Q_{12} - Q_{2}}{A_2}
\end{aligned}  \;\;\;\;\;\;\;\;\;(1)
$$
where the input flow rate is related to the control signal $0\le u \le 1$
$$
Q_i = Q_{max} u,
$$
the inter-tank flow rate is related to the height differential and $h_1 \ge h_2$ 
$$
Q_{12} = \gamma_{12} C_{d} A \sqrt{2 g (h_1 - h_2)}
$$
and the tank 2 outflow is related to the water height of that tank alone
$$
Q_{2} = \gamma_2 C_{d} A \sqrt{2 g h_2}
$$
where $\gamma_{12}$ and $\gamma_2$ are the manual valve settings [0,1] which are part of the
experiment setup.  All the valves are taken to have the same discharge coefficient $C_d$ and maximum opening area $A$.

# Implementation with *bdsim*

We start with the standard imports, create a *bdsim* simulation environment, and an empty block diagram to hold our model.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=0.2)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
r1= 0.3                # radius of first tank (m)
r2 = 0.2               # radius of second tank (m)
h = 1.0                # maximum height of tanks (m)
Qmax = 2e-3            # maximum pump flow rate (m^3/s)
g = 9.81               # gravity (m/s^2)
Cd = 0.8               # discharge coefficient
A = math.pi * 0.02**2  # area of pipe (m^2)
gamma_12 = 0.75        # manual valve setting for flow from tank 1 to tank 2
gamma_2 = 0.5          # manual valve setting for flow from tank 2 to drain

# derived parameters
A1 = math.pi * r1**2  # area of first tank (m^2)
A2 = math.pi * r2**2  # area of second tank (m^2)

As a quick sanity check on these numbers, the time to fill the first tank at full pump flow, assuming the inter-tank valve is
closed, would be over two minutes

In [ ]:
A1*h / Qmax

The system can be modeled as two integrators that represent the water heights in each tank.  The initial water levels are zero.

In [ ]:
h1 = bd.INTEGRATOR(name="h1")
h2 = bd.INTEGRATOR(name="h2")

For open-loop testing we will activate the pump to 80% capacity for 1 minute starting at $t=30$s

In [ ]:
u = bd.PIECEWISE((0, 0), (30, 0.8), (90, 0.0), name="u")  # pump command, fraction of maximum flow
Qi = bd.GAIN(Qmax, inputs=[u])

The tank outflows are a function of the water levels in the tanks and the manual valve settings

In [ ]:
Q_12 = ((2*g*(h1 - h2)) >> bd.SQRT() ) * gamma_12 *Cd * A  # flow rate from tank 1 to tank 2 (m^3/s)
Q_2 = ((2*g*h2) >> bd.SQRT() ) * gamma_2 *Cd * A  # flow rate from tank 2 to environment (m^3/s)

and these volumetric flow rates can be converted to rates of level change using the known cylindrical geometry
of the tanks which are then input to the integrators representing water levels

In [ ]:
h1[0] = (Qi - Q_12) / A1   # dh1/dt
h2[0] = (Q_12 - Q_2) / A2  # dh2/dt

To include a graphical animation of our two-tank system we need to create two functions: graphics setup,
and graphics update.  We will use simple Matplotlib
patch graphics – this kind of code is tedious to write, and it can be outsourced to a coding agent.

In [ ]:
import matplotlib.patches as patches

WATER = "#74c0ee"  # light blue
OVERFLOW = "tab:red"  # a tank that has overflowed


def tanks_init(self, fig, ax, r1=r1, r2=r2, h=h):
    # The radii and height are bound when this is defined.  Tank 1 is drawn from x=0,
    # then a gap the width of a valve, then tank 2, all standing on y=0.
    v = 0.5 * min(r1, r2)  # width of a valve, and the gap between the tanks
    vh = 0.8 * v  # height of a valve
    left = [0, 2 * r1 + v]  # left side of each tank
    width = [2 * r1, 2 * r2]

    ax.set_xlim(-v, left[1] + width[1] + 3 * v)
    ax.set_ylim(-0.05 * h, 1.15 * h)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    water = []
    for i in range(2):
        x0, x1 = left[i], left[i] + width[i]
        # the walls and floor, with no lid
        ax.plot([x0, x0, x1, x1], [h, 0, 0, h], color="black", lw=3, zorder=3)
        ax.text(x0 + width[i] / 2, 1.03 * h, f"tank {i + 1}", ha="center", va="bottom")
        # the water, with its bottom on the floor and no height until the first update
        rect = patches.Rectangle((x0, 0), width[i], 0, facecolor=WATER, edgecolor="none")
        ax.add_patch(rect)
        water.append(rect)

        # a valve at the lower right corner of the tank, a bowtie that is not animated
        cx, cy = x1 + v / 2, vh / 2
        for side in (-1, 1):
            tip = (cx, cy)
            top = (cx + side * v / 2, cy + vh / 2)
            bottom = (cx + side * v / 2, cy - vh / 2)
            ax.add_patch(patches.Polygon([tip, top, bottom], facecolor="black"))

    self.water = water
    self.h = h
    
def tanks_update(self, t, u):
    for rect, level in zip(self.water, u):  # u is the level in each tank (m)
        # a level above the top of the tank is drawn full, and red
        rect.set_height(min(max(level, 0), self.h))
        rect.set_facecolor(OVERFLOW if level > self.h else WATER)

These two functions are passed to an `ANIMATION` block driven by the outputs of the water level integrators.

In [ ]:

anim = bd.ANIMATION(
    init=tanks_init, update=tanks_update, nin=2, inputs=[h1, h2]
)

scope = bd.SCOPE(styles=("b", "r", "k--"), inputs=[h1, h2, u])


Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)
# bd.report_summary()

The model comprises 21 blocks, but the majority are were created implicitly (block names are prefixed by `_`) by the arithmetic expressions. A full list of all blocks in the model can be displayed by uncommenting the line above, and rerunning the cell. To create all these blocks and connect them would be both tedious and error prone.  The power of *bdsim* is that it lets
you move effortlessly between expression in terms of blocks or using Python expressions.

We simulate the system for four minutes, and the pump is activated at 80% capacity in the interval $t \in [30,90]$s.  We see the first tank filling quickly and then flowing through to the second tank.  After the pump is turned off, we see $h_2$ increase for a while because $h_1$ is much greater than $h_2$ and then both decline as water flows out of the system through the second valve.

Exercise: experiment with the manual valve settings $\gamma_{12}$ and $\gamma_2$ to see how the system
response changes.

In [ ]:
out = sim.run(bd, T=4*60, watch=[h1, h2, Q_12, Q_2])  # simulate for 4min
print(out)


# Understanding equilibrium conditions

For a given input flow rate the tanks will achieve an equilibrium with $h_2 \lt h_1$.  To investigate this
we will modify the *bdsim* model and replace the `PIECEWISE` block with a `CONSTANT` block

In [ ]:
bd.delete_block(u)  # delete the old input block, and ALL its connections
u = bd.CONSTANT(0.2) # create a constant value block
Qi[0] = u  # connect it to the input of the gain block
scope[2] = u  # connect it to the scope block
bd.compile()

We can simulate the new system and we see that it reaches equilibrium in around 5 minutes

In [ ]:
out = sim.run(bd, T=20*60, watch=[h1, h2])

Next, we will run a sweep to find the equilibrium levels for a range of pump settings. The simulations above have been slow because we have used graphics animation to watch the states evolve.  The actual solution time is a fraction of a second, so this sweep will be fast

In [ ]:
e = []
for pump in np.linspace(0.0, 0.9, 19): # 0 to 0.9 in steps of 0.05
    u.value = pump  # set the value of the CONSTANT block
    out = sim.run(bd, T=30*60, watch=[h1, h2], graphics=False, quiet=True)  # simulate with no graphics or console output
    e.append((pump, out.x[-1, 0], out.x[-1, 1]))  # pump, h1, h2
e = np.array(e)  # convert to a numpy array for easier plotting and interpolation

The maximum value of $u=0.90$ is chosen to avoid overflow of tank 1.

Now we can plot the equilibrium values for each tank as a function of pump command `u`

In [ ]:
import matplotlib.pyplot as plt

plt.plot(e[:,0], e[:,1], color="b", label="Tank 1")
plt.plot(e[:,0], e[:,2], color="r", label="Tank 2")
plt.xlabel("Pump flow command u (fraction of maximum)")
plt.ylabel("Equilibrium tank levels (m)")
plt.legend()
plt.xlim(0, 0.9)

From this data we can build an interpolation function, that will estimate the tank equilibrium values given 
a pump command value

In [ ]:
from scipy.interpolate import CubicSpline
xs_func = CubicSpline(e[:,0], e[:,1:])

We can evaluate it for a pump command that was not in the sweep, say $u=0.62$, and compare the result with the exact solution derived below

In [ ]:
print(xs_func(0.62))  # interpolate the equilibrium levels [h1, h2] for a pump command of 0.62

The sweep values are approximate, since the first-order dynamics have an infinite convergence time, though after 30 minutes they are within about 1% of the exact values.  The equilibrium can also be found exactly.  At equilibrium $\frac{d h_1}{dt} = \frac{d h_2}{dt} = 0$ so, from (1), $Q_{12} = Q_2 = Q_i$.  Equating the valve equation for $Q_2$ with $Q_i = Q_{max} u$ and solving for $h_2$ gives

$$
h_2 = \frac{1}{2g} \left( \frac{Q_{max} u}{\gamma_2 C_d A} \right)^2
$$

and then equating $Q_{12}$ with $Q_i$ gives

$$
h_1 = h_2 + \frac{1}{2g} \left( \frac{Q_{max} u}{\gamma_{12} C_d A} \right)^2
$$

so both levels are proportional to $u^2$.  We can compare these with the interpolated values.

In [ ]:
def equilibrium(pump):
    """Exact equilibrium tank levels [h1, h2] (m) for a pump command"""
    h2_e = (Qmax * pump / (gamma_2 * Cd * A))**2 / (2 * g)
    h1_e = h2_e + (Qmax * pump / (gamma_12 * Cd * A))**2 / (2 * g)
    return np.array([h1_e, h2_e])

print(equilibrium(0.62))  # exact solution
print(xs_func(0.62))      # interpolated from the sweep

# Linearization

We can linearize this non-linear system around an equilibrium operating point.  Firstly, we will consider the low-flow
case where $u=0.2$.  An operating point is a state $x_e$ together with the input $u$ that holds it there, so we give both to `linearize`.  Without `u` it uses the current value of the source blocks.

In [ ]:
lin1 = bd.linearize(u=0.2, x=xs_func(0.2), outputs=[h1, h2], full=True)
print(lin1)

The return value is a `Linearization` object which contains the parameters of a linear state-space model

$$
\begin{align}
\dot{\bar{x}} &= \mathbf{A} \bar{x} + \mathbf{B} \bar{u}\\
\bar{y} &= \mathbf{C} \bar{x} + \mathbf{D} \bar{u}
\end{align}
$$
where $\bar{x}$ is the state perturbation from the operating point $x_e$, $\bar{u}$ is the input
perturbation from the operating point input $u$, and $\bar{y}$ is the output
perturbation from the output value at the operating point $y$.

Note that `linearize` says this is not an equilibrium, but the offset $2\times 10^{-6}$ m/s is negligible and is due to the equilibrium states being approximate, since they were interpolated from the sweep.



The linearization objects also contain the system poles, two real poles for this system.
We can see that their positions vary considerably with the operating point

In [ ]:
lin2 = bd.linearize(u=0.8, x=xs_func(0.8),outputs=[h1, h2], full=True)
print(lin1.poles)
print(lin2.poles)


## Control

We will build an LQ state-feedback controller so that the height of the water in tank 2 follows a square-wave setpoint about the equilibrium operating point for a pump command of $u=0.6$:


In [ ]:
u_e = 0.6

The equilibrium states are

In [ ]:
x_e = xs_func(u_e)
print(x_e)

At equilibrium $h_2 \approx 0.3$ so the square wave will have a small amplitude about that value, between 0.25 and 0.35m.

The linearization is

In [ ]:
lin = bd.linearize(u=u_e, x=x_e, outputs=[h1, h2], full=True)

The *bdsim* linearization object contains the statespace matrices and the corresponding state, input and output
signal names.  For the control design we will use [Python control](https://github.com/python-control/python-control) and it has its own state-space object which
*bdsim* can export

In [ ]:
import control as ct

ss = lin.ss()
print(ss)

To design the LQ controller we need to define the $Q$ (penalizes state errors) and $R$ (penalizes control effort) matrices. $Q$ must be positive semi-definite and $R$ must be positive definite -- both are typically chosen to be diagonal. In this case we apply a high penalty to $h_2$ which is the state we wish to follow. We allow $h_1$ to be relatively unconstrained -- if it were constant then $h_2$ cannot change.

In [ ]:
Q = np.diag([1, 10_000])  # weights for [h1, h2]
R = np.diag([1]) # weight for control input u

To find the optimal state-feedback $u = -\mathbf{K}x$ we need to solve the algebraic Riccati equation

In [ ]:
K, S, E = ct.lqr(ss, Q, R)
print(K)

where `K` is the optimal state-feedback gain matrix, `S` is the solution to the algebraic Riccati equation, and `E` are the eigenvalues (poles) of the closed-loop system ($A - BK$).

To solve the algebraic Riccati equation the package uses an efficient Fortran-based package `slicot` if it is available (via the Python wrapper [slycot](https://github.com/python-control/Slycot)) otherwise it falls back to a pure-Python solution implemented in SciPy.

The gain matrix $K$ has one row per input and one column per state.

## Converting open-loop model to closed-loop

We will convert our current open-loop model into a closed-loop model.

In [ ]:
bd.delete_block(u)  # delete the old input block, and ALL its connections
bd.delete_block(scope) # delete the old scope block, and ALL its connections

h2_dmd = bd.WAVEFORM(
    wave="square", freq=1 / 240.0, unit="Hz", min=0.25, max=0.35, name="setpoint"
)  # reference water height for tank 2 (m)

# state feedback tracking controller, the gain K comes from the LQ design
u = K[0, 0] * (x_e[0] - h1) + K[0, 1] * (h2_dmd - h2) + u_e

# clip the control input to be between 0 and 1, since that is the range of the pump command
u_clip =bd.CLIP(0.0, 1.0, inputs=[u]) # connect it to the input of the gain block via a clip

Qi[0] = u_clip  # connect it to the input of the gain block
# create a new scope block to watch the tank levels, the control input and the setpoint
scope = bd.SCOPE(styles=("b", "r", "k--", "r--"), inputs=[h1, h2, u_clip, h2_dmd])

bd.compile()  # check the diagram

We set the initial states of system, the tank water levels, to their equilibrium values.  This exploits *bdsim*'s 
block parameter mechanism -- overriding the values set when the `INTEGRATOR` blocks were constructed above.

In [ ]:
h1.x0 = x_e[0]
h2.x0 = x_e[1]

Simulating the closed-loop system for 5 minutes shows the desired level-following behaviour. We can observe:
* no overshoot on $h_2$, which peaks at 0.329 m against a setpoint of 0.35 m, but a significant steady-state error: at the end of each half-cycle $h_2$ is about 0.02 m (6%) below the upper setpoint and 0.014 m (6%) above the lower one,
* the input to the plant is being clipped for a significant fraction of the time, about 35% of the 5 minutes: 25% at the maximum pump flow and 10% with the pump off.

In [ ]:
out = sim.run(bd, T=5*60, watch=[h1, h2, h2_dmd, Qi])  # simulate for 5min


## Exercises for the reader

* Quantify the error.
  * Using the simulation results in `out` compute a performance measure, perhaps RMS error.
* Investigate disturbances and model error
  * Alter the model so that the manual valve settings $\gamma_{12}$ and $\gamma_2$ are changed during the simulation
  * Rerun the simulation with different values of parameters such as $r_1$, $r_2$ or $Q_{max}$ and investigate how
    the performance measure changes.
* Improve the control performance:
  * adjust the $Q$ and $R$ matrices to get better performance
  * try a different control strategy, such as:
    * PID control, 
    * model predictive control (MPC),
    * feedforward control,
    * fuzzy control,
    * neural network control (how few hidden layers can you get away with?)